# Model Training Session

This notebook demonstrates how to configure and run an end-to-end continuous training session.

**Note**:

At this point if all cells in `01_data_preparation.ipynb` have already been run, we should have artifacts ready for training under `path/to/this/notebook/experiment`.

Otherwise either point `EXP_ROOT` to where your actual data was prepared, or run the `01_data_preparation.ipynb` end-to-end to generate synthetic data for the following cell to demo a training session

## Environment Setup & Imports

In [ ]:
# If running in a cloud/remote notebook (e.g. Databricks, Colab),
# install the wheel:
# %pip install path/to/landseg-x.y.z-py3-none-any.whl

# Or if running from a local repo clone:
# %pip install -e ..

In [ ]:
# standard imports
import pathlib
import sys
# local imports
# Attempt importing landseg directly from site-packages (wheel install).
# Fall back to local '../src' only when running inside the repository clone.
try:
    import landseg.adapters.api as api
    import landseg.notebooks as nb
except ImportError:
    repo_src = pathlib.Path.cwd().parent / 'src'
    if repo_src.exists():
        sys.path.insert(0, str(repo_src))
    import landseg.adapters.api as api
    import landseg.notebooks as nb

# Resolve experiment root path (auto-bridges Databricks widgets, env vars, or local defaults).
EXP_ROOT = nb.resolve_exp_root()
print(f'Successfully imported landseg and dependencies! Active experiment root: {EXP_ROOT}')

## Basic Training Configurations (a continuous training session)

In [ ]:
# NOTE
# Ergonomic configuration widgets for interactive training parameterization
widgets = nb.NotebookWidgets()
batch_size = widgets.int('batch_size', default=16, label='Batch Size')
patch_size = widgets.int('patch_size', default=128, label='Patch Size (px)')
max_epochs = widgets.int('max_epochs', default=5, label='Max Epochs')
model_body = widgets.dropdown('model_body', default='unet', choices=['unet', 'unetpp', 'unetppp'], label='Model Body')
device = widgets.dropdown('device', default='auto', choices=['auto', 'cuda', 'cpu'], label='Execution Device')

# init configurator
configurator = api.TrainingSessionConfigurator(EXP_ROOT)

# set data loading parameters
configurator.set_data_loading(
    batch_size=batch_size,
    patch_size=patch_size,
)

# set domain source for conditioning
configurator.set_domain_source(
    category_domain=None,
    continuous_domain=None
)

# set model architecture
configurator.set_model(
    body=model_body,
    bottleneck='conv',
    base_channel=16,
)

# set optimization
configurator.set_optimization(
    optimizer='AdamW',
    learning_rate=1e-4,
    weight_decay=1e-3,
    scheduler='CosAnneal',
)

# set engine tasks related knobs
configurator.set_tasks(
    logit_adjust_alpha=0.0,
    exclude_classes=None,
    loss_weights={
        'focal': 0.5,
        'dice': 0.5,
        'spectral': 1e-3,
        'total_var': 1e-4,
        'ecological': 0.0,
    }
)

# set runtime parameters
configurator.set_runtime(
    max_epochs=max_epochs,
    active_heads=None,
    patience_epoch=None,
    track_heads=None,
)

# configure target device if specified
configurator._cfg.execution.device = device

# run pre-flight readiness check (audits dataset manifest, GPU/CUDA VRAM headroom, model backbone)
configurator.preflight()

# run the training session
api.run(configurator.running_root_config)
